# Enunciado práctica.

### Recursos

Problemas interesantes para Aprendizaje por refuerzo
 * Gymnasium: https://gymnasium.farama.org/environments/box2d/
 * Solución del Lunar Lander con DQN: https://shiva-verma.medium.com/solving-lunar-lander-openaigym-reinforcement-learning-785675066197
 * Otra solución: https://wingedsheep.com/lunar-lander-dqn/ y https://github.com/wingedsheep/blog/blob/main/lunar-lander-dqn/lunar_lander_dqn_blog.ipynb
 * The Nature of Code: https://youtu.be/lu5ul7z4icQ
 * Librería para neuroevolución: https://pypi.org/project/nevopy/

## Instalación

%pip install gymnasium  
%pip install gymnasium[box2d] 

### Acciones adicionales

#### En macos

pip uninstall swig  
xcode-select -—install (si no se tienen ya)  
pip install swig  / sudo port install swig-python
pip install 'gymnasium[box2d]' # en zsh hay que poner las comillas  

Si se da el error [NSCheapMutableString initialize] may have been in progress in another thread when fork() was called. We cannot safely call it or ignore it in the fork() child process. Crashing instead.  
Hacer  
OBJC_DISABLE_INITIALIZE_FORK_SAFETY=YES

#### en Windows

Si da error, se debe a la falta de la versión correcta de Microsoft Visual C++ Build Tools, que es una dependencia de Box2D. Para solucionar este problema, puede seguir los siguientes pasos:  
 * Descargar Microsoft Visual C++ Build Tools desde https://visualstudio.microsoft.com/visual-cpp-build-tools/.
 * Dentro de la app, seleccione la opción "Herramientas de compilación de C++" para instalar.
 * Reinicie su sesión en Jupyter Notebook.
 * Ejecute nuevamente el comando !pip install gymnasium[box2d] en la línea de comandos de su notebook.
 * A algunos les ha funcioando instalando antes pygame.

También puede deberse a no tener swig instalado o a tener una versión incorrecta, en ese caso habrá que instalarlo con

%pip uninstall swig  
%pip install swig

Si nada funciona, la solución última es instalar anaconda y cambiar el kernel.

In [1]:
!pip install -r ../../requirements.txt --quiet

<h1 style='color:red ; text-align:center'>Apartado A</h1>

In [2]:
# prueba lunar lander por humano
# apartado A de la práctica

import gymnasium as gym

env = gym.make("LunarLander-v3", render_mode="rgb_array")

import numpy as np
import pygame
import gymnasium.utils.play

lunar_lander_keys = {
    (pygame.K_UP,): 2,
    (pygame.K_LEFT,): 1,
    (pygame.K_RIGHT,): 3,
}
#gymnasium.utils.play.play(env, zoom=3, keys_to_action=lunar_lander_keys, noop=0)

In [3]:
%%HTML
<center>
 <video 
    src="media/human_gameplay.mp4" 
    style="width: 400px; border-radius: 10px; border: 5px solid rgb(47, 221, 169); margin: auto 0;" 
    autoplay 
    muted 
    loop >
 </video>


</center>

Esta es una grabación de varios minutos jugando al juego con el teclado. Podemos ver que es bastante difícil por varios motivos:

- A veces se requieren tiempos de reacción muy rápidos.
- Solo puedes tener un motor encendido a la vez (Para usar dos tienes que alternar muy rápido).
- Solo puedes encender o apagar un motor (Si quieres menos potencia tienes que encender y apagar).

Sin embargo un agente IA no tendrá ningún inconveniente con estas dificultades así que es un juego adecuado para tratar de aplicar aprendizaje por refuerzo.

<h1 style='color:red ; text-align:center'>Apartado B</h1>

In [4]:
# agente deliberativo
# Apartado B de la práctica

# observaciones [x, y, vx, vy, ang, vang, pataiz, parader]
# acciones [nada, derecho, central, izquierdo]

# IMPORTANTE: tiene que ser creado por vosotros mismos, no vale la solución de OpenAI!

import os


def policy (observation):
    x, y, vx, vy, ang, vang, pl, pr = observation

    def stabilice_angle():
        if ang < 0:
            return 1
        else:
            return 3
        
    def drift_dir():
        dir = np.sign(x)

        if dir == 1:
            return 1
        else:
            return 3
        
    def boost():
        return 2
    
    def idle():
        return 0
    
    def get_decission(s_prob, d_prob, b_prob):

        p = np.random.uniform()

        if p < s_prob:
            return stabilice_angle()
        elif p < s_prob + d_prob:
            return drift_dir()
        elif p < s_prob + d_prob + b_prob:
            return boost()
        else:
            return idle()

    

    os.system("cls")


    if y > 1.2: # Fase entrada (+Estabilización)
        return get_decission(0.35, 0.15, 0.45)
    elif abs(x) > 0.25: # Drift lateral
        if abs(ang) < 0.2: # Queremos drift
            return get_decission(0, 0.3, 0.6)
        else:
            return get_decission(0.4, 0, 0.6) # Estabilización fuerte (Si el drift es muy fuerte o el ángulo es grande queremos estabilizar, si no queremos mantener el drift)
    else: # Fase deceleración
        if vy < -0.2: # Si vamos muy rápido queremos estabilizar
            return get_decission(0.1, 0.1, 0.45)
        else:
            return get_decission(0, 0, 0) # Si no queremos decelerar fuerte, queremos mantener la posición y el ángulo, pero si el ángulo es grande queremos estabilizar aunque vayamos lento

In [5]:
env = gym.make("LunarLander-v3", render_mode="human")
observation, info = env.reset()

terminated = False
truncated = False

while not (terminated or truncated):
    action = policy(observation)
    observation, reward, terminated, truncated, info = env.step(action)

env.reset()
env.close()

In [6]:
%%HTML
<center>
 <video
    src="media\rules_gameplay\rl-video-episode-5.mp4" 
    style="width: 400px; border-radius: 10px; border: 5px solid rgb(47, 221, 169); margin: auto 0;" 
    autoplay 
    muted 
    loop >
 </video>
</center>

<h1 style='color:red ; text-align:center'>Apartado C</h1>

In [7]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils import parameters_to_vector, vector_to_parameters

class TorchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(8, 6),
            nn.ReLU(),
            nn.Linear(6, 6),
            nn.ReLU(),
            nn.Linear(6, 4)
        )


    def forward(self, x):
        x = torch.tensor(x, dtype=torch.float32)

        with torch.no_grad():
            x = self.fc(x)
            x = F.softmax(x, dim=0)
        return x.numpy()
    


    def from_chromosome(self, chromosome):
        vector_to_parameters(torch.tensor(chromosome, dtype=torch.float32), self.parameters())
    
    def to_chromosome(self):
        return parameters_to_vector(self.parameters()).detach().numpy()
    
torch_model = TorchModel()

In [ ]:
class TorchModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Rama para los 6 valores físicos
        self.physics_branch = nn.Sequential(
            nn.Linear(6, 12),
            nn.Tanh(),
            nn.Linear(12, 8),
            nn.Tanh(),
        )
        # Rama para los sensores de contacto de las patas
        self.contact_branch = nn.Sequential(
            nn.Linear(2, 4),
            nn.Tanh(),
        )
        # Fusión final
        self.head = nn.Linear(8 + 4, 4)

        self.deterministic = False    

    def forward(self, x):
        x = torch.tensor(x, dtype=torch.float32)
        with torch.no_grad():
            physics = self.physics_branch(x[:6])
            contact = self.contact_branch(x[6:])
            merged = torch.cat([physics, contact])
            out = F.softmax(self.head(merged), dim=0)

            if self.deterministic:
            # Si queremos una política determinista elegimos la que tenga mayor activación en la capa de salida
                return torch.argmax(out).item()
            else:
            # Tambien podemos interpretar la salida del softmax como la distribución de probabilidad de elegir cada acción, (como hacíamos en la política personlizada que definimos antes).
                return np.choice(4, p=out.numpy())

        return out.numpy()
    
    def from_chromosome(self, chromosome):
        vector_to_parameters(torch.tensor(chromosome, dtype=torch.float32), self.parameters())
    
    def to_chromosome(self):
        return parameters_to_vector(self.parameters()).detach().numpy()


torch_model = TorchModel()

In [19]:
# Prueba del forward.

env = gym.make("LunarLander-v3")
observation = env.reset()[0]
torch_model.forward(observation)

array([0.1939332 , 0.18730849, 0.34773338, 0.2710249 ], dtype=float32)

In [ ]:
from MLP import MLP

class EVO_Handler:
    def __init__(self, model, environment, population_size=100):
        self.env = environment
        self.model = model 

        self.population_size = population_size

        def generate_population():
            len_chromosome = len(self.model.to_chromosome())
            return np.random.uniform(-1, 1, (self.population_size, len_chromosome))
        
        self.population = generate_population()

    def _evaluate_population(self, reward_function=None):
        fitness = np.zeros(self.population_size)

        N = 3
        for i in range(self.population_size):
            ch = self.population[i]
            self.model.from_chromosome(ch)

            for _ in range(N): # media de 5 episodios para cada individuo

                observation, info = self.env.reset()
                terminated = False
                truncated = False

                max_steps = 1000
                steps = 0

                while not (terminated or truncated) and steps < max_steps:
                    action = self.model.forward(observation)

                    observation, reward, terminated, truncated, info = self.env.step(action)

                    steps += 1

                    if reward_function:
                        fitness[i] += reward_function(observation, reward)
                    else:
                        fitness[i] += reward

                if steps >= max_steps:
                    fitness[i] -= 100 # Penalización por no terminar el episodio

        
        return fitness / N

    def sort_population(self, fitness):
        sorted_indices = np.argsort(fitness)[::-1]
        self.population = self.population[sorted_indices]
        self.fitness = fitness[sorted_indices]
        return sorted_indices

    
    def evolve(self, selection, crossover, mutation, generations=100, reward_function=None, elitism_percentage=0.3):

        best_fitness = -np.inf
        best_chromosome = None

        for gen in range(generations):
            fitness = self._evaluate_population(reward_function)
            sorted_indices = self.sort_population(fitness)

            if self.fitness[0] > best_fitness:
                best_fitness = self.fitness[0]
                best_chromosome = self.population[0]

            if gen % 10 == 0:
                print(f"Generation {gen} - Best fitness: {self.fitness[0]} - Average fitness: {np.mean(self.fitness)}")

            new_gen = []

            num_elite = int(self.population_size * elitism_percentage)
            new_gen.extend(self.population[:num_elite])

            while len(new_gen) < self.population_size:
                

                parent1 = selection(self.population, self.fitness)
                parent2 = selection(self.population, self.fitness)
                child = crossover(parent1, parent2)
                child = mutation(child)
                new_gen.append(child)

            self.population = np.array(new_gen)

        return best_chromosome, best_fitness

In [21]:
def tournament_selection(population, fitness):

    K = 3

    selected_indices = np.random.choice(len(population), K, replace=False)
    selected_fitness = fitness[selected_indices]
    winner_index = selected_indices[np.argmax(selected_fitness)]
    return population[winner_index]




def blx_alpha_crossover(parent1, parent2):

    alpha = 0.5
    child = np.zeros_like(parent1)

    for i in range(len(parent1)):
        c_min = min(parent1[i], parent2[i])
        c_max = max(parent1[i], parent2[i])
        I = c_max - c_min

        lower_bound = c_min - alpha * I
        upper_bound = c_max + alpha * I

        child[i] = np.random.uniform(lower_bound, upper_bound)

    return child

def uniform_crossover(parent1, parent2):

    p = np.random.uniform(0, 1)

    if p < 0.5:
        return parent1.copy()
    else:
        return parent2.copy()



def gaussian_mutation(chromosome, mutation_rate=0.4):
    sigma = 0.1

    new_chromosome = chromosome.copy()
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            new_chromosome[i] += np.random.normal(0, sigma)
            # reset esporádico
            if np.random.rand() < 0.02: # 2% de probabilidad de reset
                new_chromosome[i] = np.random.uniform(-1, 1)
    return new_chromosome

In [22]:
env = gym.make("LunarLander-v3")
handler = EVO_Handler(torch_model, env, population_size=30)
best_chromosome, best_fitness = handler.evolve(tournament_selection, uniform_crossover, gaussian_mutation, generations=1000)

Generation 0 - Best fitness: -106.9571845940978 - Average fitness: -363.6497166909234
Generation 10 - Best fitness: -71.9602262855928 - Average fitness: -145.47128091932382
Generation 20 - Best fitness: -95.512470868428 - Average fitness: -131.34887902370366
Generation 30 - Best fitness: -72.68396954520318 - Average fitness: -135.59444613216127
Generation 40 - Best fitness: -86.67568536083388 - Average fitness: -129.713039464197
Generation 50 - Best fitness: -28.870815022609964 - Average fitness: -110.65307644180957
Generation 60 - Best fitness: -17.240134570627834 - Average fitness: -103.47660150049666
Generation 70 - Best fitness: -7.99156089016851 - Average fitness: -82.21002255132517
Generation 80 - Best fitness: -16.648673944407566 - Average fitness: -64.70915575550296
Generation 90 - Best fitness: -15.452957574250457 - Average fitness: -83.5986275315625
Generation 100 - Best fitness: 36.46712225115697 - Average fitness: -97.1262764960238
Generation 110 - Best fitness: -18.7159758

KeyboardInterrupt: 

In [ ]:
print("Best fitness:", best_fitness)
print("Best chromosome:", best_chromosome)

Best fitness: 293.04419271422563
Best chromosome: [-0.32671222 -0.26139544 -0.57955251 -0.4405005   1.03590948  1.59886719
  1.32279047 -0.29762162 -1.38590191  0.40810826  0.429293   -1.40017277
  0.82027351  0.33298048  0.88682697 -1.02100746  0.78601991 -0.3811271
 -0.85932412 -0.14337396 -0.30269496 -1.17054779 -0.41315464 -0.17231691
 -0.33054547 -0.74707065  1.97507236  0.71343539 -1.37527578  0.01386469
  0.04724985  0.99809255 -0.27024484  1.5059653  -0.42638235  2.62752468
  0.55318047 -1.05266309  0.16353511  0.97728774 -0.23117882  0.17512278
  0.26272088 -0.43744879  1.38636229 -0.24749582  0.29674534  0.48963193
  0.59080809 -2.15914466 -1.44679456  0.17633517  0.82904888 -1.73519387
  1.32536883  0.72063236  0.38090237 -0.01609228  1.576992    1.50567025
 -1.43497312 -0.59392984  0.69738834 -1.01221208  0.79247498  0.18718602
 -0.3200652   1.36664541  0.72777272 -0.05175539  2.61090499 -0.83561324
  0.558575   -0.59263115 -0.56064991 -0.01074391  0.19135545  1.15810523
 -

In [ ]:
torch_model.from_chromosome(best_chromosome)

env = gym.make("LunarLander-v3", render_mode="human")



for _ in range(10):

    observation, info = env.reset()
    terminated = False
    truncated = False

    while not (terminated or truncated):
        output = torch_model.forward(observation)
        action = np.random.choice(len(output), p=output)
        observation, reward, terminated, truncated, info = env.step(action)

env.close()

In [ ]:
def custom_reward(observation, reward):
    x, y, vx, vy, ang, vang, pl, pr = observation

    # Recompensa por estar cerca del centro
    center_reward = -abs(x)

    # Recompensa por velocidad vertical controlada
    velocity_reward = -abs(vy) if abs(vy) > 0.5 else 0

    # Recompensa por ángulo controlado
    angle_reward = -abs(ang) if abs(ang) > 0.1 else 0

    # Recompensa por aterrizaje suave
    landing_reward = reward if pl or pr else 0

    total_reward = center_reward + velocity_reward + angle_reward + landing_reward
    return total_reward

In [ ]:
architecture = [8, 6, 4]
my_mlp = MLP(architecture)

len_chromosome = len(my_mlp.to_chromosome())

pop = np.random.uniform(-1, 1, (100, len_chromosome))

### Importante:

Después de la evolución, para poder probar la red obtenido, dejar en la varoable global model la red óptima encontrada

model = ****

In [ ]:
# neuroevolución
# Apartado C de la práctica

# construir modelo
from MLP import MLP
import numpy as np

model = MLP([8, 6, 4])
ch = np.array()
model.from_chromosome(ch)

# pasar al modelo los pesos del mejor cromosoma obtenido con nuestro AG

# definir política
def policy (observation):
    s = model.forward(observation)
    action = np.argmax(s)
    return action

82


ValueError: Chromosome legnth doesn't match architecture

In [ ]:
# prueba lunar lander por agente

import gymnasium as gym

env = gym.make("LunarLander-v3", render_mode="human")

def run ():
    #observation, info = env.reset(seed=42)
    observation, info = env.reset()
    ite = 0
    racum = 0
    while True:
        action = policy(observation)
        observation, reward, terminated, truncated, info = env.step(action)
        
        racum += reward

        if terminated or truncated:
            r = (racum+200) / 500
            print(racum, r)
            return racum
    
run()

In [ ]:
N = 10
r = 0
for _ in range(N):
    r += run()
    
print(r/N)

In [ ]:
# walker

env = gym.make("BipedalWalker-v3", render_mode="human") # hardcore=True
...